# Testing Manager

This notebook is a step-by-step walkthrough for the exact workflow you want:

1. generate an experiment plan JSON
2. load the plan
3. preview it before touching hardware
4. execute it in the lab
5. inspect the log file afterwards

It uses a dummy `VB-1-13`-style experiment by default:
- one source vial
- one-component slug
- 100 uL slug volume
- repeated identical slugs
- needle wash intended between repeats later

Read the comments in each code cell before running it.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

# This makes the notebook work whether VS Code starts the kernel in
# the repo root or inside the Notebooks folder.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Core.experiment_builder import ExperimentBuilder
from Core.experiment_context import ExperimentManager

# pandas is optional here. If it is not installed, the notebook still works.
try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

print(f"Project root: {PROJECT_ROOT}")
print(f"pandas available: {pd is not None}")

Project root: C:\Users\40299205\Documents\VictorFlow
pandas available: True


## 1. Define the experiment inputs

Edit only the values in the next cell if you want to change the dummy experiment.

This is intentionally shaped to feel like `VB-1-13`:
- all slugs are identical
- each slug uses one vial only
- each slug is 100 uL
- repeat count is easy to change

In [2]:
# Change these values to define a new experiment.
# For tomorrow, this is the main cell you will edit.

experiment_id = "vb_1_13_dummy"
description = "Dummy notebook version of VB-1-13 for manager testing"

# Source vial for the slug component.
source_module = "rack1"
source_vial = 1

# Slug definition.
n_repeats = 15
slug_volume_uL = 100
air_gap_between_uL = 5
dispense_rate_mL_min = 0.5

# This is not yet enforced automatically during execution, but it is useful
# to store now so the plan clearly states the intended run policy.
wash_policy = "needle"

# Optional free-text notes for your future self.
notes = "Benzyl alcohol leaching characterisation dummy plan"

## 2. Build the tabular rows

Each row represents one component pickup. Since this dummy experiment has one-component slugs, there is one row per slug.

Later, for true mixtures, the same `slug_id` would appear on multiple rows.

In [3]:
rows = []

for i in range(1, n_repeats + 1):
    rows.append(
        {
            "slug_id": f"{experiment_id}_slug_{i:02d}",
            "slug_order": i,
            "component_order": 1,
            "module": source_module,
            "vial": source_vial,
            "volume_uL": slug_volume_uL,
        }
    )

# This plain Python list is enough. pandas is optional.
rows[:3]

[{'slug_id': 'vb_1_13_dummy_slug_01',
  'slug_order': 1,
  'component_order': 1,
  'module': 'rack1',
  'vial': 1,
  'volume_uL': 100},
 {'slug_id': 'vb_1_13_dummy_slug_02',
  'slug_order': 2,
  'component_order': 1,
  'module': 'rack1',
  'vial': 1,
  'volume_uL': 100},
 {'slug_id': 'vb_1_13_dummy_slug_03',
  'slug_order': 3,
  'component_order': 1,
  'module': 'rack1',
  'vial': 1,
  'volume_uL': 100}]

In [4]:
if pd is None:
    print("pandas is not installed in this kernel. That is fine.")
    print(f"Number of rows prepared: {len(rows)}")
else:
    display(pd.DataFrame(rows))

,slug_id,slug_order,component_order,module,vial,volume_uL
0,vb_1_13_dummy_slug_01,1,1,rack1,1,100
1,vb_1_13_dummy_slug_02,2,1,rack1,1,100
2,vb_1_13_dummy_slug_03,3,1,rack1,1,100
3,vb_1_13_dummy_slug_04,4,1,rack1,1,100
4,vb_1_13_dummy_slug_05,5,1,rack1,1,100
5,vb_1_13_dummy_slug_06,6,1,rack1,1,100
6,vb_1_13_dummy_slug_07,7,1,rack1,1,100
7,vb_1_13_dummy_slug_08,8,1,rack1,1,100
8,vb_1_13_dummy_slug_09,9,1,rack1,1,100
9,vb_1_13_dummy_slug_10,10,1,rack1,1,100


## 3. Generate the experiment folder

This writes `plan.json` and `log.json` into `Experiments/<experiment_id>/`.

`overwrite=True` means you can rerun this cell after editing the inputs above.

In [5]:
builder = ExperimentBuilder()

result = builder.build_and_create(
    experiment_id=experiment_id,
    rows=rows,
    description=description,
    global_conditions={
        "slug_volume_uL": slug_volume_uL,
        "air_gap_between_uL": air_gap_between_uL,
        "dispense_rate_mL_min": dispense_rate_mL_min,
        "wash_policy": wash_policy,
        "notes": notes,
    },
    overwrite=True,
)

print(f"Plan written to: {result['plan_path']}")
print(f"Log written to:  {result['log_path']}")
result["summary"][:3]

Plan written to: C:\Users\40299205\Documents\VictorFlow\Experiments\vb_1_13_dummy\plan.json
Log written to:  C:\Users\40299205\Documents\VictorFlow\Experiments\vb_1_13_dummy\log.json


[{'slug_id': 'vb_1_13_dummy_slug_01',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_id': 'vb_1_13_dummy_slug_02',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_id': 'vb_1_13_dummy_slug_03',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'}]

In [6]:
# Open the generated plan so you can inspect exactly what will be loaded.
with open(result["plan_path"], "r") as f:
    plan = json.load(f)

plan

{'experiment_id': 'vb_1_13_dummy',
 'description': 'Dummy notebook version of VB-1-13 for manager testing',
 'global_conditions': {'slug_volume_uL': 100,
  'air_gap_between_uL': 5,
  'dispense_rate_mL_min': 0.5,
  'wash_policy': 'needle',
  'notes': 'Benzyl alcohol leaching characterisation dummy plan'},
 'slugs': [{'slug_id': 'vb_1_13_dummy_slug_01',
   'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
  {'slug_id': 'vb_1_13_dummy_slug_02',
   'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
  {'slug_id': 'vb_1_13_dummy_slug_03',
   'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
  {'slug_id': 'vb_1_13_dummy_slug_04',
   'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
  {'slug_id': 'vb_1_13_dummy_slug_05',
   'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
  {'slug_id': 'vb_1_13_dummy_slug_06',
   'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
 

## 4. Load and preview with `ExperimentManager`

This section is home-safe. No instruments are touched here.

In [7]:
# This loads the generated experiment folder and prepares the manager state.
manager = ExperimentManager()
manager.load_experiment(experiment_id)
manager.status()

[EXPERIMENT] vb_1_13_dummy loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] Awaiting begin_run()
Mode: experiment
Experiment: vb_1_13_dummy
State: prepared
Slug index: 0


In [8]:
# Preview gives you a compact summary of the whole planned run.
# This is the cell I would always run before executing in the lab.
preview_rows = manager.preview()
preview_rows[:5]

1. vb_1_13_dummy_slug_01 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
2. vb_1_13_dummy_slug_02 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
3. vb_1_13_dummy_slug_03 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
4. vb_1_13_dummy_slug_04 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
5. vb_1_13_dummy_slug_05 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
6. vb_1_13_dummy_slug_06 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
7. vb_1_13_dummy_slug_07 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
8. vb_1_13_dummy_slug_08 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
9. vb_1_13_dummy_slug_09 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
10. vb_1_13_dummy_slug_10 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
11. vb_1_13_dummy_slug_11 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
12. vb_1_13_dummy_slug_12 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
13. vb_1_13_dummy_slug_13 | 1 components | 100.0 uL | rack1:1 (100.0 uL)
14. vb_1_13_dummy_slug_14 | 1 components | 100.0 uL | rack1:

[{'slug_number': 1,
  'slug_id': 'vb_1_13_dummy_slug_01',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 2,
  'slug_id': 'vb_1_13_dummy_slug_02',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 3,
  'slug_id': 'vb_1_13_dummy_slug_03',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 4,
  'slug_id': 'vb_1_13_dummy_slug_04',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 5,
  'slug_id': 'vb_1_13_dummy_slug_05',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'}]

In [9]:
# Optional: pull just the next slug dict so you can inspect its exact shape.
# If you run this, the manager index advances by one.
slug = manager.get_next_slug()
slug

{'slug_id': 'vb_1_13_dummy_slug_01',
 'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]}

In [10]:
# Optional: show what remains after get_next_slug() has been called.
manager.remaining_slugs()[:3]

[{'slug_id': 'vb_1_13_dummy_slug_02',
  'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
 {'slug_id': 'vb_1_13_dummy_slug_03',
  'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]},
 {'slug_id': 'vb_1_13_dummy_slug_04',
  'reaction_plan': [{'module': 'rack1', 'vial': 1, 'volume_uL': 100.0}]}]

## 5. Lab execution pattern

Only run this section after your real hardware setup notebook cells have created a live `rsg` object.

Important:
- if you used `get_next_slug()` above for inspection, restart from a fresh manager before executing
- that avoids accidentally skipping slug 1

In [ ]:
# LAB ONLY
# Use this first. It is the safest first test tomorrow.
# It makes exactly one slug from the plan.

# manager = ExperimentManager()
# manager.load_experiment(experiment_id)
# manager.begin_run()
# slug = manager.get_next_slug()
# result = rsg.create_slug(slug)
# print(slug)
# print(result)

In [ ]:
# LAB ONLY
# Use this after the single-slug test behaves correctly.
# This runs all remaining slugs in sequence using the plan defaults.

# manager = ExperimentManager()
# manager.load_experiment(experiment_id)
# manager.begin_run()
# results = manager.run_remaining_slugs(rsg)
# results

## 6. Inspect the log file

Before execution, the log should mostly be empty apart from creation metadata.
After execution, this is where you should see `begin_run`, `slug_created`, and `all_slugs_completed` events.

In [ ]:
with open(result["log_path"], "r") as f:
    log_data = json.load(f)

log_data

In [ ]:
# This gives a quick count of recorded events.
len(log_data.get("events", []))

## 7. How to adapt this to the real `VB-1-13`

Tomorrow, you should be able to do the following with minimal edits:

1. set `experiment_id` to the real experiment name
2. set `source_module` and `source_vial` to the real stock location
3. set `n_repeats` to the real repeat count
4. rebuild the experiment folder
5. run the preview cell
6. execute one slug first
7. only then run the full sequence

In [ ]:
single_slug = {
    "slug_id": "dryrun_single_01",
    "reaction_plan": [
        {"module": "rack1", "vial": 1, "volume_uL": 100}
    ]
}

single_slug

Below I have created Mock classes for the autosampler, syringe pump and DIM, aswell as an MOCK RSG instance that accepts those mock instruments. This lets me "dry test" the experiment builder and context machine without touching my platform

In [14]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [15]:
import time
time.sleep = lambda x: None

In [16]:
class MockGilson:
    def go_into_vial(self, module, vial):
        print(f"Gilson: go_into_vial({module}, {vial})")

    def go_to_vial(self, module, vial):
        print(f"Gilson: go_to_vial({module}, {vial})")

    def go_into_dim(self):
        print("Gilson: go_into_dim()")

    def ensure_z_safe(self):
        print("Gilson: ensure_z_safe()")


class MockPump:
    def withdraw_volume(self, volume, rate):
        print(f"Pump: withdraw {volume} uL @ {rate} mL/min")

    def infuse_volume(self, volume, rate):
        print(f"Pump: infuse {volume} uL @ {rate} mL/min")

    def stop(self):
        print("Pump: STOP")


class MockDIM:
    def load(self):
        print("DIM: LOAD")

    def assert_load(self):
        print("DIM: confirmed LOAD")

    def inject(self):
        print("DIM: INJECT")

In [18]:
from Ecosystems.reaction_segment_generation import RSG   # or your actual import path

mock_g = MockGilson()
mock_p = MockPump()
mock_dim = MockDIM()

rsg = RSG(mock_g, mock_p, mock_dim)

In [19]:
manager = ExperimentManager()
manager.load_experiment(experiment_id)
manager.begin_run()

slug = manager.get_next_slug()
result = rsg.create_slug(slug)

print(result)

[EXPERIMENT] vb_1_13_dummy loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] Awaiting begin_run()
[EXPERIMENT] vb_1_13_dummy now running
[Create Slug] vb_1_13_dummy_slug_01
DIM: LOAD
DIM: confirmed LOAD
[Pickup] 100.0uL from rack1 vial 1 @ 0.05mL/min
Gilson: go_into_vial(rack1, 1)
Pump: withdraw 100.0 uL @ 0.05 mL/min
[DIM] 100.0uL dispensed in DIM @ 0.5mL/min
Gilson: go_into_dim()
Pump: infuse 100.0 uL @ 0.5 mL/min
DIM: INJECT
{'slug_id': 'vb_1_13_dummy_slug_01', 'dispensed_volume_ul': 100.0, 'num_components': 1, 'air_gap_between_ul': 5.0, 'injected': True}
